# Titre du projet – Analyse et modélisation

**Squelette conforme aux conventions de cours** (`CONVENTIONS_CODE_JEDHA.md`, §1).

Ordre canonique : chargement -> exploration -> visualisations -> valeurs manquantes -> features/target -> preprocessing (Pipeline) -> split -> entrainement -> evaluation -> interpretation -> sauvegarde.

- Problematique metier : _..._
- Problematique data : _..._
- Jeu de donnees : _..._ (source, RGPD)

In [ ]:
# ===== Imports (cellule unique, groupes : stdlib -> calcul -> viz -> sklearn -> autres) =====
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

import joblib

RANDOM_STATE = 42
TEST_SIZE = 0.2
DATA_PATH = Path("data/raw/dataset.csv")
MODEL_PATH = Path("models/model.pkl")

## Chargement des donnees

In [ ]:
df = pd.read_csv(DATA_PATH)
df.head()

## Analyse exploratoire

In [ ]:
df.info()
df.describe()

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
df["target"].value_counts().plot(kind="bar", color=["#e74c3c", "#2ecc71"])
plt.title("Distribution de la cible", fontsize=14)
plt.xlabel("target")
plt.ylabel("Nombre d'observations")
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

## Gestion des valeurs manquantes

In [ ]:
def analyse_missing_data(df):
    """Renvoie un tableau des colonnes avec valeurs manquantes, triees par pourcentage."""
    missing = df.isnull().sum()
    missing_pct = 100 * missing / len(df)

    missing_table = pd.DataFrame({
        "Colonnes": missing.index,
        "Valeurs manquantes": missing.values,
        "Pourcentage": missing_pct.values,
    })
    missing_table = missing_table[missing_table["Valeurs manquantes"] > 0] \
        .sort_values("Pourcentage", ascending=False)
    return missing_table


analyse_missing_data(df)

## Selection features / target

In [ ]:
features_to_keep = ["feat_num_1", "feat_num_2", "feat_cat_1"]
target = "target"

X = df[features_to_keep].copy()
y = df[target].copy()

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.to_list()
categorical_features = X.select_dtypes(include=["object"]).columns.to_list()

print(f"Variables numeriques : {numeric_features}")
print(f"Variables categorielles : {categorical_features}")

## Pretraitement : Pipeline + ColumnTransformer

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

## Separation Train/Test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,  # classification uniquement ; retirer pour une regression
)

print(f"Train : {len(X_train)} lignes")
print(f"Test  : {len(X_test)} lignes")

## Entrainement

Le modele est une `Pipeline(preprocessor + estimator)`. Etape estimateur nommee `classifier` ou `regressor`.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

ml_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)),
])

ml_pipeline.fit(X_train, y_train)

y_pred_train = ml_pipeline.predict(X_train)
y_pred = ml_pipeline.predict(X_test)

print(f"Accuracy train : {accuracy_score(y_train, y_pred_train):0.2%}")
print(f"Accuracy test  : {accuracy_score(y_test, y_pred):0.2%}")

## Optimisation des hyperparametres (GridSearchCV)

In [ ]:
param_grid = {
    "classifier__n_estimators": [100, 200, 300],
    "classifier__max_depth": [3, 5, 10, None],
}

grid = GridSearchCV(
    ml_pipeline, param_grid,
    cv=5, scoring="f1_weighted", n_jobs=-1, return_train_score=True,
)
grid.fit(X_train, y_train)

print(f"Meilleurs parametres : {grid.best_params_}")
print(f"Meilleur score CV : {grid.best_score_}")

results = pd.DataFrame(grid.cv_results_).nlargest(10, "mean_test_score")
best_model = grid.best_estimator_
results

## Evaluation du modele

Classification : `classification_report` + matrice de confusion.
Regression : MAE / MSE / RMSE / R2 ensemble.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

y_pred = best_model.predict(X_test)

print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="Blues")
plt.tight_layout()
plt.show()

In [ ]:
# ----- Variante regression -----
# from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score
#
# mae = mean_absolute_error(y_test, y_pred)
# mse = mean_squared_error(y_test, y_pred)
# rmse = root_mean_squared_error(y_test, y_pred)
# r2 = r2_score(y_test, y_pred)
# print(f"MAE : {mae}\nMSE : {mse}\nRMSE : {rmse}\nR2 : {r2}")

## Interpretation : feature importance

In [ ]:
clf = best_model.named_steps["classifier"]
ohe = best_model.named_steps["preprocessor"].named_transformers_["cat"].named_steps["onehot"]
cat_features = ohe.get_feature_names_out(categorical_features).tolist()
all_features = numeric_features + cat_features

importances = clf.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(12, 5))
plt.barh(range(len(importances)), importances[indices][::-1])
plt.yticks(range(len(importances)), [all_features[i] for i in indices][::-1])
plt.xlabel("Importance")
plt.title("Feature importance")
plt.tight_layout()
plt.show()

## Sauvegarde du modele

On sauvegarde la **pipeline complete** (preprocessing + modele) avec `joblib`, extension `.pkl`.

In [ ]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(best_model, MODEL_PATH)
print(f"Modele sauvegarde : {MODEL_PATH}")

# Rechargement + prediction sur une nouvelle observation
loaded_model = joblib.load(MODEL_PATH)
# new_obs = pd.DataFrame({"feat_num_1": [0.0], "feat_num_2": [0.0], "feat_cat_1": ["a"]})
# loaded_model.predict(new_obs)